# Object Detection Project

## 1. Settings

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/HoangDuonng1359/object-detection"
PROJECT_DIR = Path("/kaggle/working/object-detection")

DATASET_DIR = None

IMAGE_SIZE = 416
EPOCHS = 100
BATCH_SIZE = 32
NUM_WORKERS = 2
LR = 1e-3
WEIGHT_DECAY = 1e-4
USE_AMP = True
USE_PRETRAINED_BACKBONE = True
FREEZE_BACKBONE_STEM = False

# Set these to small values for a quick smoke test. Use 0 for full train/validation.
MAX_TRAIN_BATCHES = 0
MAX_VAL_BATCHES = 0

CONF_THRESHOLD = 0.25
NMS_THRESHOLD = 0.45

# Optional hidden/test image directory. Leave as None if you only want val predictions.
TEST_IMAGE_DIR = None
FINAL_PREDICTIONS = Path("/kaggle/working/predictions.json")


## 2. Clone Code And Install Requirements

In [ ]:
import os
import subprocess
import sys

if not PROJECT_DIR.exists():
    if "YOUR_USERNAME" in REPO_URL or "YOUR_REPO" in REPO_URL:
        raise ValueError("Edit REPO_URL in the settings cell before cloning.")
    clone_cmd = ["git", "clone"]
    clone_cmd += [REPO_URL, str(PROJECT_DIR)]
    subprocess.run(clone_cmd, check=True)
else:
    print(f"Project directory already exists: {PROJECT_DIR}")

os.chdir(PROJECT_DIR)
print("cwd:", Path.cwd())

requirements = PROJECT_DIR / "requirements.txt"
if requirements.exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)], check=True)

import torch
print("python:", sys.version)
print("torch:", torch.__version__)
print("cuda_available:", torch.cuda.is_available())


Cloning into '/kaggle/working/object-detection'...


cwd: /kaggle/working/object-detection
python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
torch: 2.10.0+cu128
cuda_available: True


## 3. Locate Kaggle Input Dataset

In [3]:
PUBLIC_DIR = Path("/kaggle/input/datasets/duong1589/object-detection/public")
TRAIN_JSON = PUBLIC_DIR / "annotations" / "train.json"
VAL_JSON = PUBLIC_DIR / "annotations" / "val.json"
TRAIN_IMAGES = PUBLIC_DIR / "train" / "images"
VAL_IMAGES = PUBLIC_DIR / "val" / "images"
EVALUATOR = PUBLIC_DIR / "tools" / "evaluate_predictions.py"

print("PUBLIC_DIR:", PUBLIC_DIR)
print("TRAIN_JSON:", TRAIN_JSON)
print("VAL_JSON:", VAL_JSON)
print("TRAIN_IMAGES:", TRAIN_IMAGES)
print("VAL_IMAGES:", VAL_IMAGES)
print("EVALUATOR:", EVALUATOR, EVALUATOR.exists())


PUBLIC_DIR: /kaggle/input/datasets/duong1589/object-detection/public
TRAIN_JSON: /kaggle/input/datasets/duong1589/object-detection/public/annotations/train.json
VAL_JSON: /kaggle/input/datasets/duong1589/object-detection/public/annotations/val.json
TRAIN_IMAGES: /kaggle/input/datasets/duong1589/object-detection/public/train/images
VAL_IMAGES: /kaggle/input/datasets/duong1589/object-detection/public/val/images
EVALUATOR: /kaggle/input/datasets/duong1589/object-detection/public/tools/evaluate_predictions.py True


## 4. Sanity Check Dataset And Code

In [4]:
import json

for split_name, json_path in [("train", TRAIN_JSON), ("val", VAL_JSON)]:
    data = json.loads(json_path.read_text(encoding="utf-8"))
    print(split_name, "images=", len(data["images"]), "annotations=", len(data["annotations"]))

subprocess.run([sys.executable, "-m", "py_compile", "train.py", "predict.py"], check=True)
subprocess.run([
    sys.executable, "-m", "utils.check_dataset",
    "--annotations", str(TRAIN_JSON),
    "--image_dir", str(TRAIN_IMAGES),
    "--image_size", str(IMAGE_SIZE),
    "--batch_size", "2",
    "--train",
], check=True)


train images= 7500 annotations= 10642
val images= 1500 annotations= 2021
dataset_size=7500
classes=['person', 'car', 'dog', 'cat', 'chair']
sample_image_shape=(3, 416, 416)
sample_image_id=img_000c52c6d12f.jpg
sample_boxes=(1, 4)
batch_image_shape=(2, 3, 416, 416)
batch_box_counts=[0, 0]


CompletedProcess(args=['/usr/bin/python3', '-m', 'utils.check_dataset', '--annotations', '/kaggle/input/datasets/duong1589/object-detection/public/annotations/train.json', '--image_dir', '/kaggle/input/datasets/duong1589/object-detection/public/train/images', '--image_size', '416', '--batch_size', '2', '--train'], returncode=0)

## 5. Train

In [5]:
from IPython.display import display
import pandas as pd

train_cmd = [
    sys.executable, "train.py",
    "--train_data", str(TRAIN_JSON),
    "--val_data", str(VAL_JSON),
    "--image_dir", str(TRAIN_IMAGES),
    "--val_image_dir", str(VAL_IMAGES),
    "--checkpoint_dir", str(PROJECT_DIR / "models"),
    "--image_size", str(IMAGE_SIZE),
    "--epochs", str(EPOCHS),
    "--batch_size", str(BATCH_SIZE),
    "--num_workers", str(NUM_WORKERS),
    "--lr", str(LR),
    "--weight_decay", str(WEIGHT_DECAY),
    "--conf_threshold", str(CONF_THRESHOLD),
    "--nms_threshold", str(NMS_THRESHOLD),
]
if USE_AMP:
    train_cmd.append("--amp")
if USE_PRETRAINED_BACKBONE:
    train_cmd.append("--pretrained_backbone")
if FREEZE_BACKBONE_STEM:
    train_cmd.append("--freeze_backbone_stem")
if MAX_TRAIN_BATCHES > 0:
    train_cmd += ["--max_train_batches", str(MAX_TRAIN_BATCHES)]
if MAX_VAL_BATCHES > 0:
    train_cmd += ["--max_val_batches", str(MAX_VAL_BATCHES)]

print("Train command:")
print(" ".join(train_cmd))
subprocess.run(train_cmd, check=True)

history_files = sorted((PROJECT_DIR / "models").glob("train_history_*.csv"), key=lambda item: item.stat().st_mtime)
if history_files:
    HISTORY_CSV = history_files[-1]
    history_df = pd.read_csv(HISTORY_CSV)
    print("History CSV:", HISTORY_CSV)
    display_columns = [
        "epoch", "lr", "train_loss", "val_loss", "val_map50", "best_map50",
        "train_box_loss", "train_objectness_loss", "train_class_loss",
        "val_box_loss", "val_objectness_loss", "val_class_loss",
    ]
    display(history_df[[col for col in display_columns if col in history_df.columns]].tail(10))
else:
    HISTORY_CSV = None
    print("No train history CSV found.")


Train command:
/usr/bin/python3 train.py --train_data /kaggle/input/datasets/duong1589/object-detection/public/annotations/train.json --val_data /kaggle/input/datasets/duong1589/object-detection/public/annotations/val.json --image_dir /kaggle/input/datasets/duong1589/object-detection/public/train/images --val_image_dir /kaggle/input/datasets/duong1589/object-detection/public/val/images --checkpoint_dir /kaggle/working/object-detection/models --image_size 416 --epochs 100 --batch_size 32 --num_workers 2 --lr 0.001 --weight_decay 0.0001 --conf_threshold 0.25 --nms_threshold 0.45 --amp --pretrained_backbone
device=cuda amp=True
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 170MB/s]


epoch=1/100 lr=0.000666667 train_loss=2.8936 val_loss=2.5327 val_map50=0.0081 best_map50=0.0081


epoch=2/100 lr=0.001 train_loss=2.4837 val_loss=2.4963 val_map50=0.0128 best_map50=0.0128


epoch=3/100 lr=0.001 train_loss=2.3180 val_loss=2.2537 val_map50=0.0329 best_map50=0.0329


...

epoch=97/100 lr=2.35829e-06 train_loss=0.3502 val_loss=1.6385 val_map50=0.4486 best_map50=0.4574


epoch=98/100 lr=1.04859e-06 train_loss=0.3464 val_loss=1.6351 val_map50=0.4416 best_map50=0.4574


epoch=99/100 lr=2.62215e-07 train_loss=0.3621 val_loss=1.6357 val_map50=0.4426 best_map50=0.4574


epoch=100/100 lr=0 train_loss=0.3422 val_loss=1.6368 val_map50=0.4430 best_map50=0.4574
best_checkpoint=/kaggle/working/object-detection/models/best.pth
history=/kaggle/working/object-detection/models/train_history_20260522_202607.csv
History CSV: /kaggle/working/object-detection/models/train_history_20260522_202607.csv


,epoch,lr,train_loss,val_loss,val_map50,best_map50,train_box_loss,train_objectness_loss,train_class_loss,val_box_loss,val_objectness_loss,val_class_loss
90,91,2.109134e-05,0.347465,1.634563,0.439214,0.457419,0.048924,0.098082,0.009524,0.180435,0.547591,0.369595
91,92,1.668957e-05,0.357373,1.636047,0.450443,0.457419,0.049650,0.103911,0.010425,0.180455,0.548492,0.370562
92,93,1.279474e-05,0.352312,1.634227,0.444802,0.457419,0.048864,0.102813,0.010358,0.180505,0.546707,0.369988
93,94,9.410912e-06,0.345101,1.635296,0.444249,0.457419,0.048474,0.097295,0.010869,0.180466,0.547373,0.371183
94,95,6.541646e-06,0.347173,1.634355,0.443784,0.457419,0.048134,0.101319,0.010370,0.180365,0.546765,0.371534
95,96,4.189949e-06,0.356907,1.638490,0.440929,0.457419,0.049507,0.104240,0.010260,0.180621,0.550281,0.370206
96,97,2.358289e-06,0.350239,1.638467,0.448556,0.457419,0.048320,0.103362,0.010558,0.180510,0.550121,0.371596
97,98,1.048587e-06,0.346440,1.635146,0.441632,0.457419,0.047831,0.100778,0.013017,0.180452,0.548560,0.368654
98,99,2.622155e-07,0.362064,1.635726,0.442588,0.457419,0.049664,0.107500,0.012484,0.180485,0.548615,0.369370
99,100,0.000000e+00,0.342185,1.636752,0.443009,0.457419,0.047068,0.101903,0.009878,0.180550,0.548471,0.371064


## 6. Predict On Validation And Evaluate

In [6]:
VAL_PREDICTIONS = PROJECT_DIR / "val_predictions.json"
predict_val_cmd = [
    sys.executable, "predict.py",
    "--image_dir", str(VAL_IMAGES),
    "--output", str(VAL_PREDICTIONS),
    "--checkpoint", str(PROJECT_DIR / "models" / "best.pth"),
    "--batch_size", str(BATCH_SIZE),
    "--conf_threshold", str(CONF_THRESHOLD),
    "--nms_threshold", str(NMS_THRESHOLD),
]
print("Predict validation command:")
print(" ".join(predict_val_cmd))
subprocess.run(predict_val_cmd, check=True)

val_predictions_data = json.loads(VAL_PREDICTIONS.read_text(encoding="utf-8"))
num_pred_boxes = sum(len(item["boxes"]) for item in val_predictions_data)
print(f"Validation predictions: images={len(val_predictions_data)} boxes={num_pred_boxes}")
print("Prediction preview:")
print(json.dumps(val_predictions_data[:3], ensure_ascii=False, indent=2))

if EVALUATOR.exists():
    VAL_SCORE = PROJECT_DIR / "val_score.json"
    subprocess.run([
        sys.executable, str(EVALUATOR),
        "--ground_truth", str(VAL_JSON),
        "--predictions", str(VAL_PREDICTIONS),
        "--output", str(VAL_SCORE),
    ], check=True)
    score = json.loads(VAL_SCORE.read_text(encoding="utf-8"))
    print("Validation score:")
    print(json.dumps(score, ensure_ascii=False, indent=2))
    if "per_class" in score:
        per_class_df = pd.DataFrame.from_dict(score["per_class"], orient="index")
        display(per_class_df)
else:
    VAL_SCORE = None
    print("Evaluator not found, skipped local validation scoring.")


Predict validation command:
/usr/bin/python3 predict.py --image_dir /kaggle/input/datasets/duong1589/object-detection/public/val/images --output /kaggle/working/object-detection/val_predictions.json --checkpoint /kaggle/working/object-detection/models/best.pth --batch_size 32 --conf_threshold 0.25 --nms_threshold 0.45
wrote 1500 predictions to /kaggle/working/object-detection/val_predictions.json
Validation predictions: images=1500 boxes=1535
Prediction preview:
[
  {
    "image_id": "img_00090df9158f.jpg",
    "boxes": []
  },
  {
    "image_id": "img_003c3ddc6a82.jpg",
    "boxes": [
      {
        "class": "dog",
        "confidence": 0.477065,
        "bbox": [
          2.26,
          95.34,
          495.62,
          362.56
        ]
      }
    ]
  },
  {
    "image_id": "img_01073356e41c.jpg",
    "boxes": []
  }
]
{
  "mAP@0.5": 0.457419,
  "performance_points": 15,
  "iou_threshold": 0.5,
  "num_ground_truth_boxes": 2021,
  "num_predictions": 1535,
  "micro_precision": 0.7

,ap,num_ground_truth,num_predictions,true_positives,false_positives,recall,precision
person,0.615844,1074,873,716,157,0.666667,0.820160
car,0.468438,283,190,144,46,0.508834,0.757895
dog,0.441941,206,190,118,72,0.572816,0.621053
cat,0.535166,176,130,105,25,0.596591,0.807692
chair,0.225705,282,152,83,69,0.294326,0.546053


## 7. Predict On Hidden/Test Images If Available

In [7]:
if TEST_IMAGE_DIR is None:
    print("TEST_IMAGE_DIR is None. Skipped final prediction.")
else:
    test_dir = Path(TEST_IMAGE_DIR)
    final_cmd = [
        sys.executable, "predict.py",
        "--image_dir", str(test_dir),
        "--output", str(FINAL_PREDICTIONS),
        "--checkpoint", str(PROJECT_DIR / "models" / "best.pth"),
        "--batch_size", str(BATCH_SIZE),
        "--conf_threshold", str(CONF_THRESHOLD),
        "--nms_threshold", str(NMS_THRESHOLD),
    ]
    print("Final predict command:")
    print(" ".join(final_cmd))
    subprocess.run(final_cmd, check=True)
    final_data = json.loads(FINAL_PREDICTIONS.read_text(encoding="utf-8"))
    print(f"Final predictions: images={len(final_data)} boxes={sum(len(item['boxes']) for item in final_data)}")
    print("Final prediction preview:")
    print(json.dumps(final_data[:3], ensure_ascii=False, indent=2))
    print("Final predictions path:", FINAL_PREDICTIONS)


TEST_IMAGE_DIR is None. Skipped final prediction.


## 8. Collect Artifacts

In [8]:
import shutil

ARTIFACT_DIR = Path("/kaggle/working/artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

files_to_copy = [
    PROJECT_DIR / "models" / "best.pth",
    PROJECT_DIR / "models" / "last.pth",
    VAL_PREDICTIONS,
    PROJECT_DIR / "val_score.json",
]
files_to_copy += sorted((PROJECT_DIR / "models").glob("train_history_*.csv"))
if FINAL_PREDICTIONS.exists():
    files_to_copy.append(FINAL_PREDICTIONS)

copied = []
for src in files_to_copy:
    if src.exists():
        dst = ARTIFACT_DIR / src.name
        shutil.copy2(src, dst)
        copied.append(dst)

zip_path = shutil.make_archive("/kaggle/working/object_detection_artifacts", "zip", ARTIFACT_DIR)
print("Artifacts directory:", ARTIFACT_DIR)
print("Artifacts zip:", zip_path)
print("Files:")
artifact_rows = []
for path in sorted(ARTIFACT_DIR.iterdir()):
    artifact_rows.append({"file": path.name, "size_mb": round(path.stat().st_size / (1024 * 1024), 3)})
display(pd.DataFrame(artifact_rows))


Artifacts directory: /kaggle/working/artifacts
Artifacts zip: /kaggle/working/object_detection_artifacts.zip
Files:


,file,size_mb
0,best.pth,191.781
1,last.pth,191.781
2,train_history_20260522_202607.csv,0.061
3,train_history_20260523_023548.csv,0.001
4,val_predictions.json,0.348
5,val_score.json,0.001


## 9. Summary Shown In Notebook

In [9]:
summary = {
    "best_checkpoint": str(PROJECT_DIR / "models" / "best.pth"),
    "history_csv": str(HISTORY_CSV) if HISTORY_CSV is not None else None,
    "val_predictions": str(VAL_PREDICTIONS),
    "val_score": str(PROJECT_DIR / "val_score.json"),
    "artifacts_zip": "/kaggle/working/object_detection_artifacts.zip",
}
print(json.dumps(summary, ensure_ascii=False, indent=2))

if HISTORY_CSV is not None:
    history_df = pd.read_csv(HISTORY_CSV)
    print("Final history row:")
    display(history_df.tail(1))

score_path = PROJECT_DIR / "val_score.json"
if score_path.exists():
    score = json.loads(score_path.read_text(encoding="utf-8"))
    print(f"mAP@0.5={score.get('mAP@0.5')} performance_points={score.get('performance_points')}")


{
  "best_checkpoint": "/kaggle/working/object-detection/models/best.pth",
  "history_csv": "/kaggle/working/object-detection/models/train_history_20260522_202607.csv",
  "val_predictions": "/kaggle/working/object-detection/val_predictions.json",
  "val_score": "/kaggle/working/object-detection/val_score.json",
  "artifacts_zip": "/kaggle/working/object_detection_artifacts.zip"
}
Final history row:


,run_started_at,epoch,total_epochs,train_data,val_data,image_dir,val_image_dir,image_size,batch_size,num_workers,...,train_objectness_loss,train_class_loss,train_num_positive,val_loss,val_box_loss,val_objectness_loss,val_class_loss,val_num_positive,val_map50,best_map50
99,2026-05-22 20:26:07,100,100,/kaggle/input/datasets/duong1589/object-detect...,/kaggle/input/datasets/duong1589/object-detect...,/kaggle/input/datasets/duong1589/object-detect...,/kaggle/input/datasets/duong1589/object-detect...,416,32,2,...,0.101903,0.009878,44.140426,1.636752,0.18055,0.548471,0.371064,42.893617,0.443009,0.457419


mAP@0.5=0.457419 performance_points=15
